# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [2]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
from collections import defaultdict
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Andersen_Downloads/"
temp_files = downloads + "Andersen_Temp_Files/"
complete_files = downloads + "Andersen_Complete_Files/"

os.chdir(downloads)

In [3]:
# Read metadata

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

print(len(metadata)) # 7397 rows

# Split date format to only check year
for date in metadata["Collection_Date"]:
    if "/" in date or date == "missing":
        metadata = metadata[metadata["Collection_Date"] != date]
metadata["Collection_Date"] = metadata["Collection_Date"].apply(lambda x: x.split("-")[0])
metadata["Collection_Date"] = metadata["Collection_Date"].apply(lambda x: int(x))

# Find only >= 2024 using run ID from metadata
metadata_new = metadata[metadata["Collection_Date"] >= 2024]
metadata_new["Collection_Date"] = metadata_new["Collection_Date"].astype(int) # Years are not floats

print(len(metadata_new)) # 6053 rows

7397
6053


C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\5\ipykernel_33068\2731231253.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_new["Collection_Date"] = metadata_new["Collection_Date"].astype(int) # Years are not floats


### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype: B3.13 or D1.1]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use output.tsv

In [4]:
# Get genotype from genoflu
os.chdir(temp_files)
output_tsv = pd.read_csv("output.tsv", delimiter="\t")

b313_and_d11_only = output_tsv[(output_tsv["Genotype"] == "B3.13") | (output_tsv["Genotype"] == "D1.1")]
b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")
# print(b313_and_d11_only)
print(len(b313_and_d11_only)) # 5160 rows

metadata_new = metadata_new.merge(b313_and_d11_only, on="Run", how="inner")

print(len(metadata_new)) # 5160

5160
5160


In [5]:
# Get animals from animal reference
os.chdir(downloads)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata_new, animals_ref) # Get host type
metadata_new["years"] = metadata_new["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

In [6]:
print(len(metadata_new))

5160


In [ ]:
# Get geolocation from genbank_mapping.tsv

os.chdir(metadata_folder)
genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
genbank_mapping = genbank_mapping.drop_duplicates(subset="Run")
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2])

metadata_genbank = metadata_new.merge(genbank_mapping, on="Run", how="inner")

print(len(metadata_genbank))
display(metadata_genbank)

3357


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,Host_Type,years,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name,name_state
0,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,avian,2024,SRR28752446_HA_cns.fa,Consensus_SRR28752446_HA_cns_threshold_0.5_qua...,SRR28752446,HA,PP740722.1,4,A/blackbird/Texas/24-008354-001/2024,Texas
1,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,2024,...,cattle,2024,SRR28752447_HA_cns.fa,Consensus_SRR28752447_HA_cns_threshold_0.5_qua...,SRR28752447,HA,PP752829.1,4,A/cattle/Texas/24-009108-005/2024,Texas
2,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,2024,...,cattle,2024,SRR28752448_HA_cns.fa,Consensus_SRR28752448_HA_cns_threshold_0.5_qua...,SRR28752448,HA,PP752821.1,4,A/cattle/Texas/24-009108-004/2024,Texas
3,SRR28752449,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,Viral,19686302,USDA-NVSL,2024,...,cattle,2024,SRR28752449_HA_cns.fa,Consensus_SRR28752449_HA_cns_threshold_0.5_qua...,SRR28752449,HA,PP752813.1,4,A/cattle/Texas/24-009108-003/2024,Texas
4,SRR28752450,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,Viral,38846827,USDA-NVSL,2024,...,cattle,2024,SRR28752450_HA_cns.fa,Consensus_SRR28752450_HA_cns_threshold_0.5_qua...,SRR28752450,HA,PP752805.1,4,A/cattle/Texas/24-009108-002/2024,Texas
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3352,SRR32653896,WGS,147.21,170346135,PRJNA1102327,SAMN47305024,Viral,58690418,USDA-NVSL,2025,...,cattle,2025,SRR32653896_HA_cns.fa,Consensus_SRR32653896_HA_cns_threshold_0.5_qua...,SRR32653896,HA,PV338836.1,4,A/cattle/CA/25-003262-003-original/2024,CA
3353,SRR32653897,WGS,148.06,69910137,PRJNA1102327,SAMN47305023,Viral,23973854,USDA-NVSL,2025,...,cattle,2025,SRR32653897_HA_cns.fa,Consensus_SRR32653897_HA_cns_threshold_0.5_qua...,SRR32653897,HA,PV338828.1,4,A/cattle/CA/25-003262-002-original/2024,CA
3354,SRR32653898,WGS,148.21,61895474,PRJNA1102327,SAMN47305022,Viral,21444614,USDA-NVSL,2025,...,cattle,2025,SRR32653898_HA_cns.fa,Consensus_SRR32653898_HA_cns_threshold_0.5_qua...,SRR32653898,HA,PV338820.1,4,A/cattle/CA/25-003262-001-original/2024,CA
3355,SRR32653899,WGS,148.15,67537431,PRJNA1102327,SAMN47305021,Viral,25457165,USDA-NVSL,2025,...,avian,2025,SRR32653899_HA_cns.fa,Consensus_SRR32653899_HA_cns_threshold_0.5_qua...,SRR32653899,HA,PV336676.1,4,A/Duck/CA/25-003063-003-original/2025,CA


In [8]:
# Get collection date from GenBank eutils 

def search_collection_date(biosample):

    print(biosample)

    try:

        # Avoid spamming the server
        time.sleep(2)
    
        base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
        search_url = base_url + "esearch.fcgi?db=biosample&term=" + biosample +"&usehistory=y&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

        # Get Biosample ID from search_url
        output = requests.get(search_url)
        xml = output.content
        root = ET.fromstring(xml)
        sample_id = root.find("./IdList/Id").text

        biosample_url = base_url + "elink.fcgi?dbfrom=biosample&db=nuccore&id=" + sample_id + "&cmd=neighbor_history&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"
        
        # Get Nucleotide ID from biosample_url
        output = requests.get(biosample_url)
        xml = output.content
        root = ET.fromstring(xml)
        query_key = root.find(".//QueryKey").text
        web_env = root.find(".//WebEnv").text

        nucleotide_url = base_url + "esummary.fcgi?db=nuccore&query_key=" + query_key + "&WebEnv=" + web_env + "&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

        output = requests.get(nucleotide_url) 
        xml = output.content
        root = ET.fromstring(xml)

        # Grab collection date at the end of the sub name
        collection_date = root.find(".//SubName").text.split("|")[-1]

        return collection_date
    
    except:
        print("Unable to find collection date.")

        if len(metadata_genbank[metadata_genbank["BioSample"] == biosample]["years"]) > 0: # If a year exists
            collection_date = metadata_genbank[metadata_genbank["BioSample"] == biosample]["years"].values[0]
        else:
            collection_date = float('nan') 

        return collection_date
    
# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["BioSample"].apply(search_collection_date)

In [9]:
# # Save this so we don't have to do it again

# os.chdir(temp_files)
# metadata_genbank.to_csv("metadata_genbank.csv")

In [10]:
# Upload saved data
os.chdir(temp_files)
metadata_genbank = pd.read_csv("metadata_genbank.csv")
display(metadata_genbank)

,Unnamed: 0,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,...,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name,name_state,Collection_Date_Specific,Name
0,0,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,...,SRR28752446_HA_cns.fa,Consensus_SRR28752446_HA_cns_threshold_0.5_qua...,SRR28752446,HA,PP740722.1,4,A/blackbird/Texas/24-008354-001/2024,Texas,16-Mar-2024,>A/Blackbird/Texas/24-008354-001-original/2024...
1,1,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,...,SRR28752447_HA_cns.fa,Consensus_SRR28752447_HA_cns_threshold_0.5_qua...,SRR28752447,HA,PP752829.1,4,A/cattle/Texas/24-009108-005/2024,Texas,20-Mar-2024,>A/Cattle/Texas/24-009108-005-original/2024|H5...
2,2,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,...,SRR28752448_HA_cns.fa,Consensus_SRR28752448_HA_cns_threshold_0.5_qua...,SRR28752448,HA,PP752821.1,4,A/cattle/Texas/24-009108-004/2024,Texas,20-Mar-2024,>A/Cattle/Texas/24-009108-004-original/2024|H5...
3,3,SRR28752449,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,Viral,19686302,USDA-NVSL,...,SRR28752449_HA_cns.fa,Consensus_SRR28752449_HA_cns_threshold_0.5_qua...,SRR28752449,HA,PP752813.1,4,A/cattle/Texas/24-009108-003/2024,Texas,20-Mar-2024,>A/Cattle/Texas/24-009108-003-original/2024|H5...
4,4,SRR28752450,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,Viral,38846827,USDA-NVSL,...,SRR28752450_HA_cns.fa,Consensus_SRR28752450_HA_cns_threshold_0.5_qua...,SRR28752450,HA,PP752805.1,4,A/cattle/Texas/24-009108-002/2024,Texas,20-Mar-2024,>A/Cattle/Texas/24-009108-002-original/2024|H5...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3352,3352,SRR32653896,WGS,147.21,170346135,PRJNA1102327,SAMN47305024,Viral,58690418,USDA-NVSL,...,SRR32653896_HA_cns.fa,Consensus_SRR32653896_HA_cns_threshold_0.5_qua...,SRR32653896,HA,PV338836.1,4,A/cattle/CA/25-003262-003-original/2024,CA,05-Nov-2024,>A/CATTLE/CA/25-003262-003/2025|H5N1|05-Nov-20...
3353,3353,SRR32653897,WGS,148.06,69910137,PRJNA1102327,SAMN47305023,Viral,23973854,USDA-NVSL,...,SRR32653897_HA_cns.fa,Consensus_SRR32653897_HA_cns_threshold_0.5_qua...,SRR32653897,HA,PV338828.1,4,A/cattle/CA/25-003262-002-original/2024,CA,05-Nov-2024,>A/CATTLE/CA/25-003262-002/2025|H5N1|05-Nov-20...
3354,3354,SRR32653898,WGS,148.21,61895474,PRJNA1102327,SAMN47305022,Viral,21444614,USDA-NVSL,...,SRR32653898_HA_cns.fa,Consensus_SRR32653898_HA_cns_threshold_0.5_qua...,SRR32653898,HA,PV338820.1,4,A/cattle/CA/25-003262-001-original/2024,CA,05-Nov-2024,>A/CATTLE/CA/25-003262-001/2025|H5N1|05-Nov-20...
3355,3355,SRR32653899,WGS,148.15,67537431,PRJNA1102327,SAMN47305021,Viral,25457165,USDA-NVSL,...,SRR32653899_HA_cns.fa,Consensus_SRR32653899_HA_cns_threshold_0.5_qua...,SRR32653899,HA,PV336676.1,4,A/Duck/CA/25-003063-003-original/2025,CA,22-Jan-2025,>A/DUCK/CA/25-003063-003/2025|H5N1|22-Jan-2025...


In [ ]:
for num, collection_date in enumerate(metadata_genbank["Collection_Date_Specific"]):
    if collection_date != collection_date: # If nan
        metadata_genbank.loc[num, "Collection_Date_Specific"] = metadata_genbank.loc[num, "years"]

# Make names

names = ">A/" + metadata_genbank["Host"] + "/" + metadata_genbank["name_state"] + "/" + metadata_genbank["isolate"] + "/" + metadata_genbank["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata_genbank["Collection_Date_Specific"] + "|" + metadata_genbank["Host_Type"] + "|" + metadata_genbank["Genotype"]

metadata_genbank["Name"] = names

# metadata_genbank.to_csv("metadata_genbank_named.csv")

display(metadata_genbank)

,Unnamed: 0,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,...,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name,name_state,Collection_Date_Specific,Name
0,0,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,...,SRR28752446_HA_cns.fa,Consensus_SRR28752446_HA_cns_threshold_0.5_qua...,SRR28752446,HA,PP740722.1,4,A/blackbird/Texas/24-008354-001/2024,Texas,16-Mar-2024,>A/Blackbird/Texas/24-008354-001-original/2024...
1,1,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,...,SRR28752447_HA_cns.fa,Consensus_SRR28752447_HA_cns_threshold_0.5_qua...,SRR28752447,HA,PP752829.1,4,A/cattle/Texas/24-009108-005/2024,Texas,20-Mar-2024,>A/Cattle/Texas/24-009108-005-original/2024|H5...
2,2,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,...,SRR28752448_HA_cns.fa,Consensus_SRR28752448_HA_cns_threshold_0.5_qua...,SRR28752448,HA,PP752821.1,4,A/cattle/Texas/24-009108-004/2024,Texas,20-Mar-2024,>A/Cattle/Texas/24-009108-004-original/2024|H5...
3,3,SRR28752449,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,Viral,19686302,USDA-NVSL,...,SRR28752449_HA_cns.fa,Consensus_SRR28752449_HA_cns_threshold_0.5_qua...,SRR28752449,HA,PP752813.1,4,A/cattle/Texas/24-009108-003/2024,Texas,20-Mar-2024,>A/Cattle/Texas/24-009108-003-original/2024|H5...
4,4,SRR28752450,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,Viral,38846827,USDA-NVSL,...,SRR28752450_HA_cns.fa,Consensus_SRR28752450_HA_cns_threshold_0.5_qua...,SRR28752450,HA,PP752805.1,4,A/cattle/Texas/24-009108-002/2024,Texas,20-Mar-2024,>A/Cattle/Texas/24-009108-002-original/2024|H5...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3352,3352,SRR32653896,WGS,147.21,170346135,PRJNA1102327,SAMN47305024,Viral,58690418,USDA-NVSL,...,SRR32653896_HA_cns.fa,Consensus_SRR32653896_HA_cns_threshold_0.5_qua...,SRR32653896,HA,PV338836.1,4,A/cattle/CA/25-003262-003-original/2024,CA,05-Nov-2024,>A/CATTLE/CA/25-003262-003/2025|H5N1|05-Nov-20...
3353,3353,SRR32653897,WGS,148.06,69910137,PRJNA1102327,SAMN47305023,Viral,23973854,USDA-NVSL,...,SRR32653897_HA_cns.fa,Consensus_SRR32653897_HA_cns_threshold_0.5_qua...,SRR32653897,HA,PV338828.1,4,A/cattle/CA/25-003262-002-original/2024,CA,05-Nov-2024,>A/CATTLE/CA/25-003262-002/2025|H5N1|05-Nov-20...
3354,3354,SRR32653898,WGS,148.21,61895474,PRJNA1102327,SAMN47305022,Viral,21444614,USDA-NVSL,...,SRR32653898_HA_cns.fa,Consensus_SRR32653898_HA_cns_threshold_0.5_qua...,SRR32653898,HA,PV338820.1,4,A/cattle/CA/25-003262-001-original/2024,CA,05-Nov-2024,>A/CATTLE/CA/25-003262-001/2025|H5N1|05-Nov-20...
3355,3355,SRR32653899,WGS,148.15,67537431,PRJNA1102327,SAMN47305021,Viral,25457165,USDA-NVSL,...,SRR32653899_HA_cns.fa,Consensus_SRR32653899_HA_cns_threshold_0.5_qua...,SRR32653899,HA,PV336676.1,4,A/Duck/CA/25-003063-003-original/2025,CA,22-Jan-2025,>A/DUCK/CA/25-003063-003/2025|H5N1|22-Jan-2025...


In [12]:
# # Make fasta files

# fasta_folder = originals + "avian-influenza/fasta/"

# os.chdir(fasta_folder)

# pairs = []
# fasta_files = {}

# for genotype in ["B3.13", "D1.1"]:
#     for segment in ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]:
#         pair = genotype + "_" + segment
#         pairs.append(pair)

# for pair in pairs:
#     fasta_files[pair] = [] # List to hold fasta files

# for run in metadata_genbank["Run"].values: # For each run 
#     for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
#         for file in files:
#             file_name = os.path.join(dirpath, file) # Get file name
#             # print(file_name)
#             if run in file_name: # Note that there will be ~8 files total with that run name
#                 # Make a fasta file and put it in the list
#                 with open(file_name) as f:
#                     lines = f.readlines()
#                     sequence = lines[1] 
#                     # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
#                     header = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Name"].values[0]
#                     genotype = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Genotype"].values[0]
#                     # print(header)
#                     # print(genotype)
#                     # break 
#                     segment = file_name.split("_")[-2]
#                     # Find the pair that corresponds to 
#                     pair_name = genotype + "_" + segment
#                     this_specific_fasta = []
#                     for pair in pairs:
#                         # print(pair)
#                         # print(pair_name)
#                         if pair_name == pair:
#                             this_specific_fasta.append(header)
#                             this_specific_fasta.append(sequence)
#                             fasta_files[pair].append(this_specific_fasta)
#                 f.close()

In [13]:
# # Create fasta files 
# os.chdir(complete_files)

# for pair in fasta_files.keys():
#     output_path = complete_files + pair + ".fasta" 

#     output_file = open(output_path, "w")
#     for item in fasta_files[pair]:
#         # for item in item:
#         # item = fasta_files[pair]
#         try:
#             name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
#         except:
#             name = str(item[0])
#         print(name)
#         # First is header, second is sequence
#         # print(value)
#         output_file.write(name + "\n")
#         output_file.write(item[1])
#     output_file.close()

In [ ]:
# De-duplication 

# Gisaid 

gisaid = downloads + "GISAID_Complete_Fasta_Files/"

os.chdir(gisaid)

def create_dataframes(directory):
    dfs_gisaid = defaultdict(list)
    for dirpath, dirs, files in os.walk(directory): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            gisaid_df = pd.DataFrame()
            with open(file_name) as f:
                lines = f.readlines()
                isolate_partial = []
                full_header = []
                sequence = []
                # Some lines start with 25_, others 25-. This shouldn't matter, but split on "_" first
                for num, line in enumerate(lines):
                    if line[0] == ">": # If it's a header
                        full_header.append(line)
                        full = line.split("/")[3] # Get the isolate
                        partial = full.split("_")[-1] # If 25_, get the last bit
                        digits = partial.split("-")
                        isolate = ""
                        other = ""
                        for d in digits:
                            # print(d)
                            if len(d) == 6 and d.isnumeric(): # If it's just digits and not one of those weird isolates
                                isolate = d + "-"
                            elif len(d) == 3 and d.isnumeric():
                                isolate = isolate + d
                            elif d.isnumeric() == False: # If it's a weird isolate
                                other = d + "-"
                            else: 
                                other = other + d
                        # Now add to list to check in Andersen files without doing wild for loops
                        if len(isolate) == 10: # If this is a correctly formatted isolate
                            # isolates.append(isolate)
                            # All headers are followed by sequences
                            isolate_partial.append(isolate)
                        else: # If this is some other isolate
                            isolate_partial.append(other)
                    elif line == "nan\n":
                        # print(directory) # Some headers in the Andersen files don't exist 
                        isolate_partial.append(float('nan'))
                        full_header.append(line) # Sorry :/
                    else: # It's a sequence
                        sequence.append(line)
                    
                gisaid_df["isolate_partial"] = isolate_partial
                gisaid_df["full_header"] = full_header
                # print(len(sequence))
                gisaid_df["sequence"] = sequence
                dfs_gisaid[file_name.split("/")[-1][:-6]].append(gisaid_df)
    return dfs_gisaid

In [23]:
dfs_gisaid = create_dataframes(gisaid)
print(dfs_gisaid["B3.13_HA"][0])

     isolate_partial                                        full_header  \
0                     >A/dairy_cow/South_Dakota/24_010354-017-300/20...   
1                     >A/dairy_cow/South_Dakota/24_010354-018-300/20...   
2                     >A/dairy_cow/South_Dakota/24_010354-019-300/20...   
3                     >A/dairy_cow/South_Dakota/24_010354-013-300/20...   
4                     >A/dairy_cow/South_Dakota/24_010354-014-300/20...   
...              ...                                                ...   
3364      008749-007  >A/dairy_cow/Texas/24-008749-007/2024|H5N1|202...   
3365      008749-006  >A/dairy_cow/Texas/24-008749-006/2024|H5N1|202...   
3366      005915-001  >A/peregrine_falcon/California/005915-001/2024...   
3367      003692-001  >A/Wild-Bird/Wyoming/24-003692-001/2024|H5N1|2...   
3368               5  >A/dairy_cow/Kansas/5/2024|H5N1|2024-04-01|cat...   

                                               sequence  
0     atggagaacatagtactacttcttgcaatagttag

In [24]:
# Do the same with Andersen 

dfs_andersen = create_dataframes(complete_files)

C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen_Complete_Files/
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen_Complete_Files/
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen_Complete_Files/
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen_Complete_Files/
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen_Complete_Files/
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen_Complete_Files/
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen_Complete_Files/
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen_Complete_Files/


In [17]:
print(dfs_andersen["B3.13_HA"][0])

     isolate_partial                                        full_header  \
0                     >A/cat/South_Dakota/001/2024|H5N1|2024-04-10|f...   
1         000003-001  >A/CHICKEN/CA/25-000003-001/2025|H5N1|28-Dec-2...   
2         000026-001  >A/dairy_cow/USA/000026-001/2025|H5N1|2025|cat...   
3         000026-002  >A/dairy_cow/USA/000026-002/2025|H5N1|2025|cat...   
4         000045-001  >A/CHICKEN/CA/25-000045-001/2025|H5N1|29-Dec-2...   
...              ...                                                ...   
3510           UW11-  >A/dairy_cow/Ohio/B24OSU-UW11-863/2024|H5N1|20...   
3511           UW12-  >A/dairy_cow/Ohio/B24OSU-UW12-871/2024|H5N1|20...   
3512           UW13-  >A/dairy_cow/Ohio/B24OSU-UW13-869/2024|H5N1|20...   
3513       original-  >A/Cattle/Michigan/24-010303-001-original-300/...   
3514             NaN                                              nan\n   

                                               sequence  
0     atggagaacatagtactacttcttgcaatagttag

In [18]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    for i, df in enumerate(dataframes):
        full_df = df.merge(dfs_gisaid[key][i], how="outer")
        # print(full_df)
        full_df = full_df.drop_duplicates(subset=["isolate_partial"])
        full_dfs[key].append(full_df)



In [19]:
print(full_dfs["B3.13_HA"][0])

     isolate_partial                                        full_header  \
0                     >A/cat/South_Dakota/001/2024|H5N1|2024-04-10|f...   
22        000003-001  >A/CHICKEN/CA/25-000003-001/2025|H5N1|28-Dec-2...   
25        000026-001  >A/dairy_cow/USA/000026-001/2025|H5N1|2025|cat...   
26        000026-002  >A/dairy_cow/USA/000026-002/2025|H5N1|2025|cat...   
27        000045-001  >A/CHICKEN/CA/25-000045-001/2025|H5N1|29-Dec-2...   
...              ...                                                ...   
5817           UW11-  >A/dairy_cow/Ohio/B24OSU-UW11-863/2024|H5N1|20...   
5818           UW12-  >A/dairy_cow/Ohio/B24OSU-UW12-871/2024|H5N1|20...   
5819           UW13-  >A/dairy_cow/Ohio/B24OSU-UW13-869/2024|H5N1|20...   
5820       original-  >A/Cattle/Michigan/24-010303-001-original-300/...   
5821             NaN                                              nan\n   

                                               sequence  
0     atggagaacatagtactacttcttgcaatagttag

In [20]:
# Create fasta files 
os.chdir(complete_files)
for pair in full_dfs.keys():
    output_path = complete_files + pair + ".fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
        # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()

In [21]:

# unique_animals_all = sort_animals_anderson(metadata_new)

# # Flatten unique_animals
# every_unique_animal = []
# for animal in unique_animals_all:
#     every_unique_animal.append(animal)

# print(every_unique_animal)

# unique_animals_set = list(set(every_unique_animal))
# # animals_df = pd.DataFrame(columns=["avian", "cattle", "feline", "other_mammal", "human", "other"])
# # animals_df["other"] = unique_animals_set # to sort

# os.chdir(downloads)

# animals_ref = pd.read_csv("animals_ref.csv")


# # If animal not in ref1, put in ref2

# common_animals = []
# # Check if animals in unique_animals_set are in ref1
# for animal in unique_animals_set:
#     for col in animals_ref.columns:
#         if animal in animals_ref[col].values and type(animal) == str:
#             common_animals.append(animal)

# print(common_animals)
# print(len(common_animals))

# different_animals = []
# for animal in unique_animals_set:
#     if animal not in common_animals:
#         different_animals.append(animal)

# print(different_animals)

# # Add to dataframe
# animals_df = animals_ref
# # Make different_animals same length as dataframe, if shorter
# if len(different_animals) < len(animals_df):
#     number_of_times_to_add_nan = len(animals_df) - len(different_animals)
#     for i in range(number_of_times_to_add_nan):
#         different_animals.append(float('nan'))
# # If longer, deal with that later

# animals_df["new"] = (different_animals)

# print(animals_df)

# animals_df.to_csv("animals_ref_to_sort.csv")
